# 2. Averages → aligned, centered holograms
Estimate shifts from your supplied averages using masked FFT cross-correlation and subpixel refinement. The alignment routine is local to this workflow. The reference observation is a **zero-based array index**. The red `mask_pixel.png` must match the raw detector grid: red means excluded.

Alignment is estimated on the raw grid. Apply alignment and optional detector centering together, once, using linear interpolation without periodic wrapping. Move exclusion masks conservatively and mark the exposed boundaries invalid. The support PNG is supplied later on the final, centered object grid.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import h5py
import matplotlib.pyplot as plt

HERE = Path.cwd().resolve()
if HERE.name != "permanent_magnets":
    HERE = HERE / "permanent_magnets"
REPO = HERE.parent
sys.path.insert(0, str(REPO))
from permanent_magnets import workflow


In [ ]:
PROCESSED = Path("/path/to/processed")  # Edit: same output folder in all four notebooks.
from scipy.ndimage import shift, maximum_filter
INPUT = PROCESSED / "01_averages.h5"
OUTPUT = PROCESSED / "02_holograms.h5"
MASK_PNG = HERE / "masks" / "mask_pixel.png"  # Red = excluded, raw detector grid.
REFERENCE_INDEX = 0
MAX_SHIFT = 10.0
UPSAMPLE_FACTOR = 20
ALIGNMENT_ROI = None  # REQUIRED: write (row_start, row_stop, column_start, column_stop), raw pixels.
DIAGNOSTIC_INDICES = [0, 1]  # Choose two distinct, zero-based observation indices for the plots.
DETECTOR_CENTER = None  # Or measured (row, column); None keeps the original center.
CROP = 0  # Equal edge crop AFTER shifting; support PNG must match this final shape.

## Inspect the full detector and choose two observations


In [ ]:
with h5py.File(INPUT, "r") as h:
    images = h["averages"][()]
    scan_indices = h["scan_indices"][()]
    energies = h["energy_eV"][()]
    polarizations = h["polarization"][()]
    frame_counts = h["frame_counts"][()]
painted_mask = workflow.red_mask(MASK_PNG, images.shape[-2:])
masks = np.broadcast_to(painted_mask, images.shape).copy() | ~np.isfinite(images)
assert 0 <= REFERENCE_INDEX < len(images)
assert isinstance(CROP, int) and 0 <= 2*CROP < min(images.shape[-2:])
a, b = DIAGNOSTIC_INDICES
assert a != b and 0 <= a < len(images) and 0 <= b < len(images)
pair_mask = masks[a] | masks[b]
raw_difference = np.where(pair_mask, np.nan, images[a] - images[b])
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, image, title in zip(axes,
        [np.where(masks[a], np.nan, images[a]), np.where(masks[b], np.nan, images[b]), raw_difference],
        [f"Hologram {a}, scan {scan_indices[a]}", f"Hologram {b}, scan {scan_indices[b]}", f"Difference {a} - {b}"]):
    artist = ax.imshow(image, cmap="RdBu_r" if ax is axes[2] else "viridis")
    ax.set(title=title, xlabel="Column (raw pixels)", ylabel="Row (raw pixels)")
    fig.colorbar(artist, ax=ax, shrink=.75)
plt.show()


## Select and inspect the cross-correlation region

You must set `ALIGNMENT_ROI` yourself. Use a region with stable fringes/contrast and enough unmasked pixels; avoid the beamstop and boundaries. Coordinates are raw detector pixels, with exclusive stop indices. This region is used to estimate every correction against `REFERENCE_INDEX`.


In [ ]:
# Choose this region yourself, using the full-detector plots above.
# Edit ALIGNMENT_ROI in the settings cell and rerun it before continuing.
if ALIGNMENT_ROI is None:
    raise ValueError("Set ALIGNMENT_ROI = (row_start, row_stop, column_start, column_stop) before aligning")
assert len(ALIGNMENT_ROI) == 4 and all(isinstance(v, (int, np.integer)) for v in ALIGNMENT_ROI)
r0, r1, c0, c1 = ALIGNMENT_ROI
assert 0 <= r0 < r1 <= images.shape[1] and 0 <= c0 < c1 <= images.shape[2]
roi = np.s_[r0:r1, c0:c1]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, image, title in zip(axes,
        [np.where(masks[a], np.nan, images[a])[roi], np.where(masks[b], np.nan, images[b])[roi], raw_difference[roi]],
        [f"Hologram {a} in alignment region", f"Hologram {b} in alignment region", f"Difference {a} - {b} in alignment region"]):
    artist = ax.imshow(image, extent=(c0, c1, r1, r0), cmap="RdBu_r" if ax is axes[2] else "viridis")
    ax.set(title=title, xlabel="Column (raw pixels)", ylabel="Row (raw pixels)")
    fig.colorbar(artist, ax=ax, shrink=.75)
plt.show()


## Estimate shifts, apply corrections, and save


In [ ]:
shifts = np.zeros((len(images), 2))
scores = np.ones((len(images), 2))
for i in range(len(images)):
    if i != REFERENCE_INDEX:
        shifts[i], scores[i] = workflow.estimate_hologram_shift(
            images[REFERENCE_INDEX], images[i], masks[REFERENCE_INDEX], masks[i],
            max_shift=MAX_SHIFT, upsample_factor=UPSAMPLE_FACTOR, roi=ALIGNMENT_ROI)
    print(f"Scan {scan_indices[i]}: corrective row/column shift {shifts[i]}, correlation {scores[i]}")

centering_shift = np.zeros(2) if DETECTOR_CENTER is None else np.asarray(images.shape[-2:])//2 - np.asarray(DETECTOR_CENTER)
corrected = np.empty_like(images)
corrected_masks = np.empty_like(masks)
for i in range(len(images)):
    delta = shifts[i] + centering_shift
    corrected[i] = shift(np.where(masks[i], 0, images[i]), delta, order=1, mode="constant", cval=0, prefilter=False)
    # Linear interpolation touches neighbouring pixels: dilation protects their footprint.
    guarded_mask = maximum_filter(masks[i].astype(float), size=3) if np.any(delta != np.rint(delta)) else masks[i].astype(float)
    corrected_masks[i] = shift(guarded_mask, delta, order=1, mode="constant", cval=1, prefilter=False) > 0
region = np.s_[CROP:-CROP, CROP:-CROP] if CROP else np.s_[:, :]
corrected = corrected[:, region[0], region[1]]
corrected_masks = corrected_masks[:, region[0], region[1]] | ~np.isfinite(corrected) | (corrected < 0)
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
with h5py.File(OUTPUT, "w") as h:
    h.attrs["source_file"] = str(INPUT.resolve())
    h.attrs["mask_png"] = str(MASK_PNG.resolve())
    h.attrs["reference_index"] = REFERENCE_INDEX
    h.attrs["crop"] = CROP
    h.attrs["max_shift"] = MAX_SHIFT
    h.attrs["upsample_factor"] = UPSAMPLE_FACTOR
    h.attrs["alignment_roi"] = "full detector" if ALIGNMENT_ROI is None else str(ALIGNMENT_ROI)
    h.attrs["shift_convention"] = "corrective row, column; raw detector pixels"
    for name, array in [("holograms", corrected), ("mask_pixel", corrected_masks),
                        ("scan_indices", scan_indices), ("energy_eV", energies),
                        ("polarization", polarizations), ("frame_counts", frame_counts),
                        ("alignment_shifts", shifts), ("alignment_correlation", scores),
                        ("centering_shift", centering_shift)]:
        h.create_dataset(name, data=array, compression="gzip")
print("Saved", OUTPUT)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].plot(shifts[:, 0], "o-", label="Row"); axes[0].plot(shifts[:, 1], "o-", label="Column")
axes[0].set(xlabel="Observation", ylabel="Corrective shift (raw pixels)"); axes[0].legend()
axes[1].plot(scores[:, 0], "o-", label="Before"); axes[1].plot(scores[:, 1], "o-", label="After")
axes[1].set(xlabel="Observation", ylabel="Normalized correlation"); axes[1].legend()
plt.show()
reference = images[REFERENCE_INDEX]
workflow.plot_stack(np.where(masks | masks[REFERENCE_INDEX], np.nan, images-reference), scan_indices, "Before alignment: difference from reference", "RdBu_r")
reference = corrected[REFERENCE_INDEX]
workflow.plot_stack(np.where(corrected_masks | corrected_masks[REFERENCE_INDEX], np.nan, corrected-reference), scan_indices, "After alignment/centering: difference from reference", "RdBu_r")
workflow.plot_stack(np.log1p(np.maximum(corrected, 0)), scan_indices, "Corrected holograms")
workflow.plot_stack(corrected_masks, scan_indices, "Final exclusion masks", "gray", 0, 1)

## Shifted pair difference and its FTH reconstruction


In [ ]:
# Compare the same two observations AFTER alignment, centering, and cropping.
# Hide excluded pixels in the difference; replace them by zero only for the FFT.
from library.fthcore import reconstruct
pair_mask = corrected_masks[a] | corrected_masks[b]
shifted_difference = corrected[a] - corrected[b]
fth_difference = reconstruct(np.where(pair_mask, 0, shifted_difference))
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
limit = np.nanpercentile(abs(np.where(pair_mask, np.nan, shifted_difference)), 99)
limit = max(float(limit), 1e-12)
artist = axes[0].imshow(np.where(pair_mask, np.nan, shifted_difference), cmap="RdBu_r", vmin=-limit, vmax=limit)
axes[0].set_title(f"Shifted hologram difference {a} - {b}")
fig.colorbar(artist, ax=axes[0], shrink=.75)
artist = axes[1].imshow(np.log1p(abs(fth_difference)))
axes[1].set_title("FTH of shifted difference: log(1 + amplitude)")
fig.colorbar(artist, ax=axes[1], shrink=.75)
plt.show()
# Preserve the diagnostic pair and reconstruction alongside the holograms.
with h5py.File(OUTPUT, "a") as h:
    h.create_dataset("diagnostic_indices", data=DIAGNOSTIC_INDICES)
    h.create_dataset("diagnostic_difference", data=shifted_difference, compression="gzip")
    h.create_dataset("diagnostic_difference_mask", data=pair_mask, compression="gzip")
    h.create_dataset("diagnostic_fth_difference", data=fth_difference, compression="gzip")
